This script plots various aspects of the TrappedResonance objective function

In [1]:
import desc.io
from desc.objectives import (
    TrappedResonance
)
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

In [10]:
# User Inputs #
eq = desc.io.load("equil_G1600_DESC_fixed.h5")
rhos = (np.linspace(0.1,0.9,50))**(1/2) # rho = sqrt(s)
alphas = np.linspace(0,2*np.pi,5)
KE_frac = np.array([0.0001]) #did 0.001 before
pitch_invs = jnp.array([6.05])
N=0 # QA

In [11]:
# Run objective function
eq_periodicity = (np.inf,np.inf,np.inf) # periodicity in zeta for these equilibrium to make rtz grid
grid = eq._get_rtz_grid( # returns rho, theta, zeta coordinate grid
    rhos, # radial
    np.array([0]), # poloidal (alpha in this case)
    np.array([0]), # toroidal (zeta in this case)
    coordinates="raz", # rho, alpha, zeta input coordinates
    period=eq_periodicity, # periodicity of coordinate (rho,alpha,zeta)
)
Psi = eq.compute("Psi",grid=grid)

obj = TrappedResonance(eq,rho=rhos,pitch_invs=pitch_invs,KE_frac=KE_frac,alpha=alphas,N=N,Psi=Psi['Psi'])
# obj = TrappedResonance(eq,rho=rhos,pitch_invs=pitch_invs,KE_frac=KE_frac,alpha=alphas,N=N)
obj.build()
out = obj.compute(eq.params_dict) # := (rho,lambda,well)

Precomputing transforms


In [12]:
# Poincare plotting saving
poinc_plot = np.sum(out['poinc_plot'],axis=2)[:,0] # := (rho), summed over wells cuz why not
omega_arr = out['omega_arr'][:,0,0]

# Save objective function values to the firm3d directory for plotting with Poincare plots
np.savetxt('/Users/paullab/codes/firm3d_fork_10132025/firm3d/examples/trapped_map/obj_DESC.txt',poinc_plot)
np.savetxt('/Users/paullab/codes/firm3d_fork_10132025/firm3d/examples/trapped_map/freq_DESC.txt',omega_arr)
# np.savetxt('/Users/paullab/codes/firm3d_fork_10132025/firm3d/examples/trapped_map/tau_DESC.txt',out['tau_arr'][:,0,0,0])

In [9]:
print(out['pitch_inv'])

[[5.95]
 [5.95]
 [5.95]
 [5.95]
 [5.95]
 [5.95]
 [5.95]
 [5.95]
 [5.95]
 [5.95]]


In [ ]:
# Plot s vs. obj_out for each pitch inverse
for i in range(0,obj_out.shape[1]): # loop through each pitch angle
    plt.figure(i+1)
    plt.plot(rhos**2,obj_out[:,i,0]) # assuming we are only doing 1 energy for now
    plt.xlabel(r'$s$')
    plt.ylabel('Objective Function Value')
    plt.title('TrappedResonance Objective Function Value for Pitch Inverse = '+str(pitch_invs[i]))
    # plt.ylabel(r'$\omega_{\zeta}$')
    # plt.title(r'$\omega_{\zeta}$ Value for Pitch Inverse = '+str(pitch_invs[i]))
    plt.xlim([0.0,1.0])

In [ ]:
# Plot the value of the TrappedResonance objective (summing over the rho axis) vs. pitch
obj_out = out['obj']
num_valid_pitch = jnp.sum(obj_out!=0,axis=0) # for each surface, how many pitches were valid

obj_out_rhosum = np.sum(obj_out,axis=0) / num_valid_pitch
np.savetxt('obj_pitch_plot.txt',obj_out_rhosum[:,0])
plt.figure()
plt.plot(pitch_invs,obj_out_rhosum[:,0]/np.nanmax(obj_out_rhosum[:,0]))
plt.title("TrappedResonance Objective Function Averaged Over All Surfaces")
plt.xlabel("Pitch Inverse [T]")
plt.ylabel("Objective Function Value")

In [51]:
np.savetxt('obj_s_out.txt',out['obj'][:,0,0]) # value at all surfaces

In [ ]:
obj_val = out['obj'][:,0,0]
Y = rhos**2
X = np.linspace(0,np.pi,5)
Z = np.transpose(np.tile(obj_val, (5, 1)))
plt.contourf(X,Y,Z)
plt.colorbar(label='Objective Function Value')